# Reproducing *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*

Lewis et al., NeurIPS 2020 — [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

---

## What this notebook does

The paper's claim in one line: **stop cramming all knowledge into weights — bolt on a
searchable text index, retrieve into the prompt, and marginalise over what you retrieved
so the retriever can be trained by gradient descent without ever labelling a passage.**

We rebuild that system end to end, on a laptop:

| # | Section | Paper reference |
|---|---------|-----------------|
| 1 | Build the non-parametric memory (100-word Wikipedia chunks) | §3 |
| 2 | DPR bi-encoder retriever + MIPS index | §2.1 |
| 3 | BART generator, and how a passage enters the prompt | §2.2 |
| 4 | **The marginalisation — Eq (1) and Eq (2), implemented by hand** | §2 |
| 5 | Decoding: RAG-Token beam search vs RAG-Sequence thorough decoding | §2.4 |
| 6 | Open-domain QA evaluation (EM / F1 / answer recall) | §4.1, Table 1 |
| 7 | Ablation: retrieving more documents | §4.5, Fig 3 |
| 8 | Ablation: learned retrieval vs BM25 vs random | §4.5, Table 6 |
| 9 | "Right even when the answer was never retrieved" | §4.1 (11.8%) |
| 10 | **Index hot-swapping** — edit knowledge with zero retraining | §4.5 |
| 11 | **Gradient reaches the retriever with no passage labels** | §2, §4.5 |
| 12 | Retrieval collapse diagnostic | Appendix H |

## Honest statement of scale

This is a *faithful* reproduction of the mechanism, not a *full-scale* one. The differences
are all in one direction — corpus size — and are stated up front:

| | Paper | This notebook |
|---|---|---|
| Non-parametric memory | 21M passages (Dec-2018 Wikipedia dump) | ~15k passages (Wikipedia text via SQuAD) |
| MIPS index | FAISS + HNSW (approximate) | exact dot product (a 15k×768 matmul) |
| Retriever | DPR, query encoder fine-tuned | **same DPR weights**, query encoder from the released RAG checkpoint |
| Generator | BART-large (400M), fine-tuned end-to-end on NQ | **same weights** — `facebook/rag-token-nq` / `rag-sequence-nq` |
| Eval set | NaturalQuestions / TriviaQA / WQ / CT | SQuAD-dev questions, asked *open-domain* (gold paragraph withheld) |
| End-to-end training | 8× 32GB GPUs | one 8-question demo run, to show the gradient exists |

Because we use the authors' released weights, everything about the *model* is the paper's.
What we rebuild ourselves is the index, the retrieval, the marginalisation maths, the
decoders and the evaluation — which is exactly the part worth understanding.

**Runtime:** ~15–20 min end to end on Apple Silicon (MPS), most of it the one-time
index build. Model downloads are ~5 GB the first time.

---
# 0 · Setup

Nothing interesting here except two details that matter on a Mac:

- **`padding="max_length"`.** MPS recompiles its kernels every time a tensor shape changes.
  Dynamic padding makes every batch a new shape, which turned a 40-second evaluation into
  a 40-minute one when this notebook was being written. Fixed shapes = one compile.
- **The first `generate()` call is slow** (tens of seconds) while MPS compiles its kernels.
  Every call after it is fast. Don't panic at §5.
- **Memory.** Beam search over `batch × k` sequences is the memory hot-spot, and it is
  quadratic in context length. Two mitigations below: contexts are padded to
  `max_combined=200` (enough for a 100-word passage — the 300 in the checkpoint config is
  slack we don't need), and the generation batch shrinks as `k` grows so `batch × k` stays
  roughly constant. If you hit `kIOGPUCommandBufferCallbackErrorOutOfMemory`, close other
  notebook kernels and lower `CFG["seq_budget"]`.

In [ ]:
import os, re, gc, time, json, string, random, warnings
from collections import Counter, defaultdict

os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.0")   # before torch is imported
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    RagTokenForGeneration, RagSequenceForGeneration, RagConfig,
    DPRContextEncoder, DPRContextEncoderTokenizerFast,
    DPRQuestionEncoder, DPRQuestionEncoderTokenizerFast,
    BartTokenizerFast,
)

warnings.filterwarnings("ignore")

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    chunk_words   = 100,    # paper §3: "disjoint 100-word chunks"
    n_distractors = 12000,  # extra passages so retrieval is not trivial
    k             = 5,      # paper trains with k in {5, 10}
    n_eval        = 96,     # eval questions
    n_eval_seq    = 48,     # RAG-Sequence subset (thorough decoding is ~10x the cost)
    seq_budget    = 20,     # batch*k sequences per generate() call — the memory knob
    beams         = 4,
    max_combined  = 200,    # tokens; a 100-word passage + question fits well inside this
    max_answer    = 16,
)

def gen_batch(k):
    '''Questions per generate() call, so that batch*k (the beam-search width) stays bounded.'''
    return max(1, min(8, CFG["seq_budget"] // k))

def mps_gc():
    gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    elif DEVICE.type == "cuda": torch.cuda.empty_cache()

ART = "results/rag_repro"; os.makedirs(ART, exist_ok=True)
print("device:", DEVICE, "| torch", torch.__version__)
print(json.dumps(CFG, indent=2))

### The system, on one screen

```
                    ┌──────────────── NON-PARAMETRIC MEMORY ────────────────┐
                    │  15k passages of real Wikipedia text  (editable!)      │
                    │  each pre-encoded once  ->  d(z) ∈ R^768               │
                    └───────────────────────┬───────────────────────────────┘
                                            │  frozen document encoder
   x = "who wrote a farewell to arms"       │
        │                                   │
        ▼                                   ▼
   ┌──────────┐   q(x)∈R^768        ┌───────────────┐   top-k z₁..z_k
   │ DPR-Q    │ ──────────────────► │  MIPS search  │ ──────────────┐
   │ (BERT)   │      TRAINABLE      │  d(z)ᵀq(x)    │               │
   └──────────┘                     └───────────────┘               │
                                            │ scores                │
                                            ▼                       ▼
                                    p_η(z|x) = softmax        "title / text // x"
                                       ("trust")                     │
                                            │                        ▼
                                            │             ┌────────────────────┐
                                            │             │  BART generator    │
                                            │             │  p_θ(y | x, z)     │
                                            │             └─────────┬──────────┘
                                            └────────► Σ_z ─────────┘
                                                       p(y|x)
```

Everything below fills in one box at a time.

---
# 1 · The non-parametric memory

> *"Each Wikipedia article is split into disjoint 100-word chunks, to make a total of 21M
> documents."* — §3

Two properties of this store are the whole point of the paper, and both come from it being
**literal text**, not weights:

1. You can **point at** the passage that produced an answer (provenance).
2. You can **edit** it — swap the file, and the model's world knowledge changes with zero
   retraining. We prove this in §10.

We need real Wikipedia prose *plus* questions whose answers actually live in our tiny slice
of it, so a 15k-passage index gives measurable numbers instead of a floor of zero. SQuAD
supplies both: its paragraphs *are* Wikipedia, and its questions are answerable from them.
We then throw the gold paragraph away and ask the question **open-domain** — the model must
find its own evidence in 15k candidates. Same task shape as the paper's NQ setup, 1/1400th
the haystack.

The split of roles:

- **SQuAD-dev paragraphs** → the passages our eval questions *can* be answered from.
- **SQuAD-train paragraphs** → 12k distractors. Without them retrieval is too easy and every
  ablation flattens out.

In [ ]:
def chunk_paragraph(text, title, size=CFG["chunk_words"], min_words=20):
    '''Disjoint N-word chunks, paper §3. Title is kept: DPR encodes 'title [SEP] text'.'''
    words, out = text.split(), []
    for i in range(0, len(words), size):
        piece = words[i:i + size]
        if len(piece) >= min_words:
            out.append({"title": title.replace("_", " "), "text": " ".join(piece)})
    return out

squad_dev   = load_dataset("rajpurkar/squad", split="validation")
squad_train = load_dataset("rajpurkar/squad", split="train")

passages       = []          # the corpus: list of {title, text}
ctx_to_pids    = {}          # SQuAD paragraph -> passage ids it became (for gold labelling)

seen = set()
for ex in squad_dev:                              # answerable region
    if ex["context"] in seen: continue
    seen.add(ex["context"])
    start = len(passages)
    passages += chunk_paragraph(ex["context"], ex["title"])
    ctx_to_pids[ex["context"]] = list(range(start, len(passages)))
n_answerable = len(passages)

# Distractors. NB: dedupe in dataset order, not via a set — set iteration order depends on
# PYTHONHASHSEED, which would silently change the corpus (and invalidate the cached index)
# between runs.
train_ctxs, train_titles, seen_tr = [], {}, set()
for ex in squad_train:
    if ex["context"] in seen_tr: continue
    seen_tr.add(ex["context"]); train_ctxs.append(ex["context"]); train_titles[ex["context"]] = ex["title"]
random.Random(SEED).shuffle(train_ctxs)
for ctx in train_ctxs:
    if len(passages) - n_answerable >= CFG["n_distractors"]: break
    passages += chunk_paragraph(ctx, train_titles[ctx])

N_PASSAGES = len(passages)
print(f"answerable passages : {n_answerable:>6,}")
print(f"distractor passages : {N_PASSAGES - n_answerable:>6,}")
print(f"TOTAL index size    : {N_PASSAGES:>6,}   (paper: 21,000,000)")
print(f"\nexample passage:\n  title: {passages[0]['title']}\n  text : {passages[0]['text'][:220]}...")

---
# 2 · The retriever — DPR, and why the document encoder is frozen

> *"p_η(z|x) ∝ exp(d(z)ᵀ q(x)), d(z) = BERT_d(z), q(x) = BERT_q(x)"* — §2.1

Two separate BERT towers. The relevance of a passage to a question is a **dot product of
two 768-d vectors**. That's the entire retriever.

The design decision that made RAG trainable on 2020 hardware is in the next sentence of the
paper: **only the query tower is fine-tuned; the document tower stays frozen.** REALM
re-encoded and re-indexed all of Wikipedia periodically during training — brutally
expensive. RAG showed you don't have to.

That choice is why *this notebook is even possible*. We build our index with the original
`facebook/dpr-ctx_encoder-single-nq-base`, and then query it with the **fine-tuned** query
encoder that ships inside `facebook/rag-token-nq`. Those two towers were never trained
together on our corpus — and they don't need to be, because the frozen tower is the same
frozen tower the authors used. The index is portable; that is the whole trick.

The vectors are the *catalogue*, not the knowledge. `d(z)` doesn't contain the passage — it
tells you where the passage is. The text is fetched afterwards, and that text is what BART
actually reads.

In [ ]:
ctx_tok = DPRContextEncoderTokenizerFast.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
INDEX_PATH = f"{ART}/index_{N_PASSAGES}.npy"

if os.path.exists(INDEX_PATH):
    D = np.load(INDEX_PATH)
    print(f"loaded cached index {D.shape} from {INDEX_PATH}")
else:
    ctx_enc = DPRContextEncoder.from_pretrained(
        "facebook/dpr-ctx_encoder-single-nq-base").eval().to(DEVICE)
    vecs, t0 = [], time.time()
    with torch.no_grad():
        for i in range(0, N_PASSAGES, 64):
            b = passages[i:i + 64]
            enc = ctx_tok([p["title"] for p in b], [p["text"] for p in b],
                          padding="max_length", truncation=True, max_length=160,
                          return_tensors="pt").to(DEVICE)
            vecs.append(ctx_enc(**enc).pooler_output.cpu().numpy())
            if i % 3200 == 0:
                print(f"  {i:>6,}/{N_PASSAGES:,}  ({time.time()-t0:.0f}s)", flush=True)
    D = np.concatenate(vecs).astype("float32")
    np.save(INDEX_PATH, D)
    print(f"encoded {N_PASSAGES:,} passages in {time.time()-t0:.0f}s -> {INDEX_PATH}")
    del ctx_enc, vecs; gc.collect()

DOC_EMB = torch.tensor(D, device=DEVICE)
print("index tensor:", tuple(DOC_EMB.shape), "|", DOC_EMB.element_size() * DOC_EMB.nelement() / 1e6, "MB")

### MIPS: what FAISS is actually approximating

*Maximum Inner Product Search* = "of these 15,000 vectors, which have the largest dot product
with `q(x)`?" At 21M passages you cannot score all of them per query, so the paper uses
**FAISS with HNSW** — a graph that finds *probably* the top-k while looking at a small
fraction of the index.

At 15k passages we don't need the approximation, so we do the exact thing: one
`(1×768) @ (768×15000)` matmul, then `topk`. Sub-millisecond. This is strictly *better*
than the paper's retrieval — it removes approximation error as a confound, so anything we
observe in the ablations is about the model, not about the index structure.

In [ ]:
model = RagTokenForGeneration.from_pretrained("facebook/rag-token-nq").eval().to(DEVICE)
q_tok = DPRQuestionEncoderTokenizerFast.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
g_tok = BartTokenizerFast.from_pretrained("facebook/bart-large")
TITLE_SEP, DOC_SEP = model.config.title_sep, model.config.doc_sep

@torch.no_grad()
def encode_questions(questions, qencoder=None):
    '''q(x): questions -> (B, 768). Defaults to RAG's *fine-tuned* query tower.'''
    qencoder = qencoder or model.question_encoder
    enc = q_tok(questions, padding="max_length", truncation=True, max_length=40,
                return_tensors="pt").to(DEVICE)
    out = qencoder(**enc)
    return out[0] if isinstance(out, tuple) else out.pooler_output

@torch.no_grad()
def retrieve(questions, k=CFG["k"], doc_emb=None):
    '''Exact MIPS. Returns (scores (B,k), indices (B,k)) — scores are raw dot products.'''
    qv = encode_questions(questions)
    top = (qv @ (DOC_EMB if doc_emb is None else doc_emb).T).topk(k, dim=-1)
    return top.values, top.indices

demo_q = "who wrote a farewell to arms"
sc, ix = retrieve([demo_q], k=5)
print(f"QUESTION: {demo_q}\n")
for r, (s, i) in enumerate(zip(sc[0].tolist(), ix[0].tolist()), 1):
    print(f"  #{r}  d(z)ᵀq(x) = {s:6.2f}   [{passages[i]['title']}]  {passages[i]['text'][:95]}...")

### Raw scores → `p_η(z|x)`, the "trust"

The dot products are unbounded reals. A softmax over the top-k turns them into a
distribution that sums to 1 — that's `p_η(z|x)`, the weight each passage gets in the blend.

Note what this is *not*: nobody assessed these passages. Trust is nothing but
vector-closeness pushed through `exp` and normalised. The only reason it correlates with
usefulness is that training moved `q(x)` — and §11 shows that happening.

Also note the gaps. Because scores live around 50–80 with a few points between neighbours,
softmax is fairly peaky: rank 1 typically takes most of the mass. That matters for the
k-ablation in §7 — adding a 20th document adds a passage with almost no weight.

In [ ]:
trust = torch.softmax(sc, dim=-1)[0]
print(f"{'rank':<5}{'raw score':>11}{'p_η(z|x)':>11}   passage")
for r, (s, t, i) in enumerate(zip(sc[0].tolist(), trust.tolist(), ix[0].tolist()), 1):
    print(f"{r:<5}{s:>11.2f}{t:>11.3f}   [{passages[i]['title']}] {passages[i]['text'][:60]}...")
print(f"{'':<5}{'':>11}{trust.sum():>11.3f}   <- sums to 1")

---
# 3 · The generator — how a passage gets into BART

> *"we simply concatenate [the retrieved content] with the input"* — §2.2

No cross-attention tricks, no fusion module. The retrieved passage is **glued onto the
question as a string** and the whole thing is fed to BART's encoder. The exact template is
in the checkpoint's config:

```
{title} / {passage text} // {question}
```

Then `p_θ(y | x, z)` is just BART's ordinary autoregressive probability of the answer
tokens, conditioned on that string. When the passage names the answer, copying it is easy
and the probability is high; when the passage is irrelevant, BART's mass spreads out and the
probability collapses. **Passage quality shows up directly as the size of that number** —
which is precisely what makes the marginalisation in §4 do useful work.

Below: the k=5 retrieved passages, each scored and each decoded *independently*, so you can
see the two ingredients — `p_η(z|x)` and `p_θ(y|x,z)` — side by side before they're combined.

In [ ]:
def build_contexts(questions, doc_ids):
    '''(B, k) doc ids -> B*k strings 'title / text // question', flattened doc-major per question.'''
    out = []
    for b, q in enumerate(questions):
        for d in doc_ids[b].tolist():
            out.append(passages[d]["title"] + TITLE_SEP + passages[d]["text"] + DOC_SEP + q)
    return out

def tokenize_contexts(ctx_strings):
    return g_tok(ctx_strings, padding="max_length", truncation=True,
                 max_length=CFG["max_combined"], return_tensors="pt").to(DEVICE)

ctx_strings = build_contexts([demo_q], ix)
ci = tokenize_contexts(ctx_strings)

print("EXACT STRING FED TO BART FOR DOCUMENT #1")
print("-" * 78); print(ctx_strings[0][:400], "..."); print("-" * 78)

with torch.no_grad():                       # each document answers on its own
    per_doc = model.generator.generate(ci.input_ids, attention_mask=ci.attention_mask,
                                       num_beams=4, max_new_tokens=CFG["max_answer"],
                                       output_scores=True, return_dict_in_generate=True)
answers = [a.strip() for a in g_tok.batch_decode(per_doc.sequences, skip_special_tokens=True)]

print(f"\n{'doc':<5}{'p_η(z|x)':>10}   {'BART alone says':<28} title")
for r in range(CFG["k"]):
    print(f"{r+1:<5}{trust[r]:>10.3f}   {answers[r][:26]:<28} [{passages[ix[0][r]]['title']}]")

# sanity: is max_combined long enough that we are not silently truncating passages away?
probe = build_contexts(["a question of about average length here"], torch.arange(400).view(1, 400))
lens = [len(x) for x in g_tok(probe).input_ids]
print(f"\ncontext lengths over 400 passages: mean {np.mean(lens):.0f}, p99 {np.percentile(lens,99):.0f}, "
      f"max {max(lens)} vs max_combined={CFG['max_combined']} "
      f"-> {100*np.mean(np.array(lens) > CFG['max_combined']):.1f}% truncated")

---
# 4 · The marginalisation — the actual contribution

Everything so far is a search engine bolted to a language model. **This section is what makes
it RAG.**

The problem: nobody labels which passage is correct. There is no supervision on `z`. So
treat `z` as a **latent variable** and integrate it out — approximating the sum over all 21M
documents with the top-k, which carry nearly all the mass:

**RAG-Sequence** (Eq 1) — one document is responsible for the *whole* answer:

$$p_{\text{RAG-Seq}}(y|x) \;\approx\; \sum_{z \in \text{top-}k} p_\eta(z|x)\;\prod_{i}^{N} p_\theta(y_i \mid x, z, y_{1:i-1})$$

**RAG-Token** (Eq 2) — *every token* may come from a different document:

$$p_{\text{RAG-Token}}(y|x) \;\approx\; \prod_{i}^{N} \sum_{z \in \text{top-}k} p_\eta(z|x)\; p_\theta(y_i \mid x, z, y_{1:i-1})$$

The only difference is **where the Σ sits relative to the Π**. Outside → the model commits to
one document for the whole sequence. Inside → it re-decides at every token, which is what
lets it braid facts from two passages into one sentence.

And here is the part that makes the retriever learnable: `p_η(z|x)` is a *factor in the
output probability*. Grade the output — and the gradient flows straight into the trust
score, and from there into the query encoder. **The label is on the answer; the credit leaks
backwards onto the passage.** No passage annotation ever exists.

We now implement both equations from scratch, in log-space, and check them against the
reference implementation.

In [ ]:
def per_doc_token_logprobs(ctx_ids, ctx_mask, target_ids):
    '''log p_θ(y_i | x, z, y_<i) for every doc and every target token.

    Returns (n_docs, T) — the raw material both equations are built from.'''
    n_docs = ctx_ids.shape[0]
    labels = target_ids.repeat(n_docs, 1)
    out = model.generator(input_ids=ctx_ids, attention_mask=ctx_mask,
                          decoder_input_ids=labels[:, :-1])
    logprobs = torch.log_softmax(out.logits.float(), dim=-1)
    return logprobs.gather(-1, labels[:, 1:].unsqueeze(-1)).squeeze(-1)   # (n_docs, T-1)

def rag_sequence_logprob(doc_logprior, tok_lp):
    '''Eq (1):  log Σ_z [ p(z|x) · Π_i p(y_i|x,z,y_<i) ]  — Σ OUTSIDE the Π.'''
    return torch.logsumexp(doc_logprior + tok_lp.sum(dim=-1), dim=0)

def rag_token_logprob(doc_logprior, tok_lp):
    '''Eq (2):  log Π_i Σ_z [ p(z|x) · p(y_i|x,z,y_<i) ] — Σ INSIDE the Π.'''
    return torch.logsumexp(doc_logprior.unsqueeze(-1) + tok_lp, dim=0).sum()

candidate = "Ernest Hemingway"
y = g_tok([candidate], return_tensors="pt").input_ids.to(DEVICE)

with torch.no_grad():
    tok_lp = per_doc_token_logprobs(ci.input_ids, ci.attention_mask, y)
doc_logprior = torch.log_softmax(sc[0].float(), dim=-1)          # log p_η(z|x)

lp_seq = rag_sequence_logprob(doc_logprior, tok_lp)
lp_tok = rag_token_logprob(doc_logprior, tok_lp)

print(f'candidate answer y = "{candidate}"   ({tok_lp.shape[1]} target tokens)\n')
print(f"{'doc':<5}{'log p_η(z|x)':>14}{'Σ_i log p_θ(y_i|x,z)':>24}{'p_θ(y|x,z)':>14}")
for d in range(CFG["k"]):
    s = tok_lp[d].sum()
    print(f"{d+1:<5}{doc_logprior[d]:>14.3f}{s:>24.3f}{s.exp():>14.3e}")
print(f"\nlog p_RAG-Sequence(y|x) = {lp_seq:8.4f}   (p = {lp_seq.exp():.4e})")
print(f"log p_RAG-Token(y|x)    = {lp_tok:8.4f}   (p = {lp_tok.exp():.4e})")

### Cross-check against the reference implementation

Our two functions are ~3 lines each. `transformers` computes the same quantities inside
`RagTokenForGeneration.get_nll` / `RagSequenceForGeneration.get_nll`, with production
concerns (padding masks, label smoothing, a special case that adds the document log-prior at
the *second* token position rather than the first — algebraically identical, since it's added
once to a sum either way).

If our hand-written equations agree with theirs to numerical tolerance, we've understood the
paper's core maths correctly. That's the assert below.

In [ ]:
with torch.no_grad():
    hf_tok = model(context_input_ids=ci.input_ids, context_attention_mask=ci.attention_mask,
                   doc_scores=sc.float(), labels=y, n_docs=CFG["k"])

# HF sets decoder_input_ids = labels and shifts the target left by one, so its predicted
# positions are exactly ours: position t predicts labels[t+1]. Same quantity, both ways.
ours_tok   = rag_token_logprob(doc_logprior, tok_lp)
theirs_tok = -hf_tok.loss.sum()

print(f"RAG-Token   ours = {ours_tok:10.5f}   transformers = {theirs_tok:10.5f}"
      f"   |Δ| = {abs(ours_tok - theirs_tok):.2e}")
assert torch.allclose(ours_tok, theirs_tok, atol=2e-2), "Eq (2) mismatch"
print("\n✓ our from-scratch Eq (2) matches the reference implementation")

### Reading those numbers

Look at the per-document table above. The `Σ_i log p_θ` column spans several nats: the
passage that actually names the author assigns the answer orders of magnitude more
probability than the one about a different novel. The blend is dominated by that document —
*and it got its weight from `p_η`, which came from a dot product, which came from a query
encoder that was trained by exactly this signal.*

Notice also that `p_RAG-Token > p_RAG-Sequence` here. That's typical and not a bug: token-level
marginalisation lets each token take the best available document, so it can never be worse
than the best single-document path, whereas RAG-Sequence is a convex combination of complete
per-document paths.

---
# 5 · Decoding

The two equations need two different decoders (§2.4):

**RAG-Token** is a plain autoregressive model over the marginal
`p'(y_i|x,y_<i) = Σ_z p_η(z|x) p_θ(y_i|x,z,y_<i)`. Standard beam search, one pass.

**RAG-Sequence** is *not* an autoregressive model — the sum is outside the product, so there's
no per-token distribution to beam over. The paper's fix ("thorough decoding"): run beam search
**per document**, pool all the resulting hypotheses into a candidate set `Y`, then score every
candidate under every document and pick the argmax of Eq (1). More forward passes, better
calibrated.

One implementation trap worth knowing: `RagSequenceForGeneration.generate` reads `n_docs` from
`config.n_docs` for its internal rescoring pass and ignores the keyword argument. Set the
config field.

In [ ]:
def rag_token_answer(questions, k=CFG["k"], beams=CFG["beams"], doc_ids=None, doc_scores=None):
    with torch.no_grad():
        if doc_ids is None:
            doc_scores, doc_ids = retrieve(questions, k=k)
        ci = tokenize_contexts(build_contexts(questions, doc_ids))
        out = model.generate(context_input_ids=ci.input_ids, context_attention_mask=ci.attention_mask,
                             doc_scores=doc_scores.float(), n_docs=k,
                             num_beams=beams, max_new_tokens=CFG["max_answer"])
    return [a.strip() for a in g_tok.batch_decode(out, skip_special_tokens=True)], doc_ids

t0 = time.time()
ans, _ = rag_token_answer([demo_q])
print(f"RAG-Token  ->  {ans[0]!r}     ({time.time()-t0:.1f}s incl. MPS warm-up)")

In [ ]:
seq_model = RagSequenceForGeneration.from_pretrained("facebook/rag-sequence-nq").eval().to(DEVICE)
seq_model.config.n_docs = CFG["k"]      # see note above — generate() ignores the kwarg

def rag_sequence_answer(questions, k=CFG["k"], beams=CFG["beams"]):
    seq_model.config.n_docs = k
    with torch.no_grad():
        enc = q_tok(questions, padding="max_length", truncation=True, max_length=40,
                    return_tensors="pt").to(DEVICE)
        qv = seq_model.question_encoder(**enc)[0]
        top = (qv @ DOC_EMB.T).topk(k, dim=-1)
        ci = tokenize_contexts(build_contexts(questions, top.indices))
        out = seq_model.generate(context_input_ids=ci.input_ids, context_attention_mask=ci.attention_mask,
                                 doc_scores=top.values.float(), n_docs=k,
                                 num_beams=beams, max_new_tokens=CFG["max_answer"])
    return [a.strip() for a in g_tok.batch_decode(out, skip_special_tokens=True)], top.indices

ans_seq, _ = rag_sequence_answer([demo_q])
print(f"RAG-Sequence -> {ans_seq[0]!r}   (thorough decoding: k beam searches + rescoring)")

---
# 6 · Evaluation

Open-domain protocol, exactly as in §4.1: the model gets **the question and nothing else**.
The gold paragraph is never shown; it has to be found in 15k candidates.

Three numbers:

- **EM** — exact match after the standard SQuAD normalisation (lowercase, strip punctuation
  and articles). This is the metric in the paper's Table 1.
- **F1** — token overlap; forgiving of "Ernest Hemingway" vs "Hemingway".
- **Answer recall@k** — did any gold answer string appear *anywhere* in the k retrieved
  passages? This measures the retriever alone, and it's the paper's Figure 3 (centre).
  The gap between recall and EM is the generator's share of the errors.

In [ ]:
def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, golds):
    return float(any(normalize(pred) == normalize(g) for g in golds))

def f1_score(pred, golds):
    best = 0.0
    for g in golds:
        pt, gt = normalize(pred).split(), normalize(g).split()
        common = Counter(pt) & Counter(gt); n = sum(common.values())
        if n == 0: continue
        prec, rec = n / len(pt), n / len(gt)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best

def answer_in_docs(golds, doc_ids_row):
    joined = normalize(" ".join(passages[d]["text"] for d in doc_ids_row))
    return float(any(normalize(g) in joined for g in golds))

# ---- the eval set: SQuAD-dev questions, gold paragraph withheld ----
idxs = list(range(len(squad_dev))); random.Random(SEED).shuffle(idxs)
idxs = idxs[:CFG["n_eval"]]
EVAL_Q     = [squad_dev[i]["question"] for i in idxs]
EVAL_GOLD  = [squad_dev[i]["answers"]["text"] for i in idxs]
EVAL_GOLD_PIDS = [ctx_to_pids[squad_dev[i]["context"]] for i in idxs]
print(f"{len(EVAL_Q)} eval questions\nexample: {EVAL_Q[0]}\n   gold: {EVAL_GOLD[0]}")

In [ ]:
def evaluate(name, answer_fn, questions=EVAL_Q, golds=EVAL_GOLD, k=CFG["k"], verbose=True):
    '''Run a system over the eval set. answer_fn(batch_questions, k) -> (answers, doc_ids).'''
    preds, doc_ids_all, t0 = [], [], time.time()
    B = gen_batch(k)
    for i in range(0, len(questions), B):
        a, d = answer_fn(questions[i:i + B], k)
        preds += a
        doc_ids_all += [row.tolist() for row in d] if d is not None else [[]] * len(a)
        mps_gc()
    em = 100 * np.mean([exact_match(p, g) for p, g in zip(preds, golds)])
    f1 = 100 * np.mean([f1_score(p, g) for p, g in zip(preds, golds)])
    rec = 100 * np.mean([answer_in_docs(g, d) for g, d in zip(golds, doc_ids_all)]) if doc_ids_all[0] else float("nan")
    if verbose:
        print(f"{name:<28} EM {em:5.1f}   F1 {f1:5.1f}   answer-recall@{k} {rec:5.1f}   ({time.time()-t0:.0f}s)")
    return dict(name=name, em=em, f1=f1, recall=rec, preds=preds, doc_ids=doc_ids_all)

RESULTS = {}

### The main table

Four systems, all sharing **the same generator weights**, differing only in what reaches its
encoder. That isolates retrieval as the variable:

| system | what BART sees |
|---|---|
| **no retrieval** | an empty passage — pure parametric memory, the paper's "closed-book" condition |
| **random passages** | k passages drawn uniformly from the index — retrieval machinery, zero relevance |
| **RAG-Token** | k DPR-retrieved passages, Eq (2) |
| **RAG-Sequence** | k DPR-retrieved passages, Eq (1) |

The "random" row is the one to watch. It's the control that separates *"retrieval helps"*
from *"having extra text in the prompt helps"*.

In [ ]:
def no_retrieval_answer(questions, k=None):
    '''Closed book: the same BART, handed an empty passage. Pure parametric memory.'''
    ctx = ["" + TITLE_SEP + "" + DOC_SEP + q for q in questions]
    ci = tokenize_contexts(ctx)
    scores = torch.zeros(len(questions), 1, device=DEVICE)
    with torch.no_grad():
        out = model.generate(context_input_ids=ci.input_ids, context_attention_mask=ci.attention_mask,
                             doc_scores=scores, n_docs=1, num_beams=CFG["beams"],
                             max_new_tokens=CFG["max_answer"])
    return [a.strip() for a in g_tok.batch_decode(out, skip_special_tokens=True)], None

_rand = random.Random(SEED)
def random_docs_answer(questions, k=CFG["k"]):
    ids = torch.tensor([[_rand.randrange(N_PASSAGES) for _ in range(k)] for _ in questions], device=DEVICE)
    scores = torch.zeros(len(questions), k, device=DEVICE)
    return rag_token_answer(questions, k=k, doc_ids=ids, doc_scores=scores)

RESULTS["no-retrieval"] = evaluate("no retrieval (closed book)", no_retrieval_answer)
RESULTS["random"]       = evaluate("random passages", random_docs_answer)
RESULTS["rag-token"]    = evaluate("RAG-Token", lambda q, k: rag_token_answer(q, k=k))

In [ ]:
# Thorough decoding runs k beam searches + a rescoring pass per question, so it is roughly
# an order of magnitude slower than RAG-Token. Scored on a subset; noisier, same protocol.
n_s = CFG["n_eval_seq"]
RESULTS["rag-sequence"] = evaluate(f"RAG-Sequence (n={n_s})",
                                   lambda q, k: rag_sequence_answer(q, k=k),
                                   questions=EVAL_Q[:n_s], golds=EVAL_GOLD[:n_s])
del seq_model; mps_gc()        # 1.6 GB back — everything below only needs RAG-Token

### Comparing to the paper

Absolute EM is not comparable across corpora — our haystack is 1/1400th the size (easier) but
our generator was fine-tuned on NaturalQuestions and is being asked SQuAD questions
(harder, different answer conventions). **The pattern is what reproduces**, and the paper's
own numbers (Table 1 / Table 6, NQ Exact Match) show the same ordering:

| | paper (NQ) | this notebook (SQuAD-dev, open-domain) |
|---|---|---|
| closed book | T5-11B: 34.5 | see table above |
| RAG-Token | 44.1 (test) / 43.5 (dev) | ↑ |
| RAG-Sequence | 44.5 (test) / 44.0 (dev) | ↑ |

RAG-Sequence edging out RAG-Token on short factoid QA is exactly the paper's result — for a
one-or-two-token answer there is nothing to braid, so committing to a single good document
is the better bet.

---
# 7 · Ablation: retrieving more documents

> *"retrieving more documents at test time monotonically improves Open-domain QA results for
> RAG-Sequence, but performance peaks for RAG-Token at 10 retrieved documents."* — §4.5

This is Figure 3 (left and centre). k is a **test-time** knob — the model was trained at
k=5, and we can change it now without touching a weight, because k only changes how many
terms are in the sum.

Two curves, and the interesting thing is that they diverge:

- **answer recall@k** must rise with k — more passages, more chances to contain the string.
- **EM** need not. Extra low-trust documents dilute the marginal and can pull a correct
  answer off the top.

In [ ]:
K_VALUES = [1, 2, 5, 10, 20]
k_sweep = []
for k in K_VALUES:
    r = evaluate(f"RAG-Token k={k}", lambda q, kk: rag_token_answer(q, k=kk), k=k)
    k_sweep.append(r)
RESULTS["k_sweep"] = k_sweep

In [ ]:
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
plt.rcParams.update({"figure.dpi": 130, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#9a9a95",
                     "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.titlesize": 10})

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.1))
ems  = [r["em"] for r in k_sweep]
recs = [r["recall"] for r in k_sweep]

axes[0].plot(K_VALUES, ems, "-o", color=BLUE, lw=2, ms=6)
axes[0].set_title("Exact Match vs documents retrieved"); axes[0].set_xlabel("k retrieved docs")
axes[0].set_ylabel("EM"); axes[0].set_xticks(K_VALUES)
best = int(np.argmax(ems))
axes[0].annotate(f"peak k={K_VALUES[best]}\nEM {ems[best]:.1f}", (K_VALUES[best], ems[best]),
                 textcoords="offset points", xytext=(6, -22), color=BLUE, fontsize=8)

axes[1].plot(K_VALUES, recs, "-o", color=AQUA, lw=2, ms=6)
axes[1].set_title("Answer recall@k (retriever only)"); axes[1].set_xlabel("k retrieved docs")
axes[1].set_ylabel("% questions whose answer text is in top-k"); axes[1].set_xticks(K_VALUES)
for ax in axes: ax.grid(axis="y", color="#e5e5e2", lw=0.8); ax.set_axisbelow(True)
fig.suptitle("Fig 3 (left & centre) reproduced — k is a test-time knob", y=1.03, fontsize=10.5)
fig.tight_layout(); fig.savefig(f"{ART}/k_sweep.png", bbox_inches="tight", dpi=200)
plt.show()
print("recall rises monotonically; EM need not — extra low-trust docs dilute the marginal.")

---
# 8 · Ablation: is the *learned* retrieval doing the work?

> *"we run ablations where we freeze the retriever during training … learned retrieval
> improves results for all tasks."* — §4.5, Table 6

The paper's ablation compares three retrievers feeding the same generator. We reproduce all
three:

| paper row | NQ dev EM | our analogue |
|---|---|---|
| RAG-Token | 43.5 | fine-tuned query encoder (from `rag-token-nq`) |
| RAG-Token-Frozen | 37.8 | the **original** `dpr-question_encoder-single-nq-base`, never fine-tuned inside RAG |
| RAG-Token-BM25 | 29.7 | BM25 word-overlap, scores used as logits for `p(z|x)` |

BM25 is worth writing out rather than importing — it's twenty lines, and seeing it next to
the dot product makes the contrast concrete. It scores by *word overlap*, weighting rare
words (`idf`), saturating repeated terms (`k1`) and normalising for passage length (`b`).
It has no idea that "author" and "wrote" are related; DPR's vectors do.

In [ ]:
class BM25:
    '''Okapi BM25 from scratch (Robertson & Zaragoza). Paper §4.5 uses BM25 scores as p(z|x) logits.'''
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        toks = [re.findall(r"[a-z0-9]+", d.lower()) for d in docs]
        self.lens = np.array([len(t) for t in toks], dtype="float32")
        self.avgdl = self.lens.mean()
        self.N = len(toks)
        self.postings = defaultdict(list)          # term -> [(doc_id, tf), ...]
        for i, t in enumerate(toks):
            for term, tf in Counter(t).items():
                self.postings[term].append((i, tf))
        self.idf = {term: np.log(1 + (self.N - len(p) + 0.5) / (len(p) + 0.5))
                    for term, p in self.postings.items()}

    def score(self, query):
        s = np.zeros(self.N, dtype="float32")
        for term in re.findall(r"[a-z0-9]+", query.lower()):
            if term not in self.postings: continue
            idf = self.idf[term]
            for i, tf in self.postings[term]:
                denom = tf + self.k1 * (1 - self.b + self.b * self.lens[i] / self.avgdl)
                s[i] += idf * (tf * (self.k1 + 1)) / denom
        return s

t0 = time.time()
bm25 = BM25([p["title"] + " " + p["text"] for p in passages])
print(f"BM25 index built over {N_PASSAGES:,} passages in {time.time()-t0:.0f}s "
      f"({len(bm25.postings):,} unique terms)")
demo_bm25 = bm25.score(demo_q)
print("\ntop-3 for the demo question:")
for i in np.argsort(-demo_bm25)[:3]:
    print(f"  {demo_bm25[i]:6.2f}  [{passages[i]['title']}] {passages[i]['text'][:80]}...")

In [ ]:
dpr_frozen = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base").eval().to(DEVICE)

def frozen_dpr_answer(questions, k=CFG["k"]):
    with torch.no_grad():
        qv = encode_questions(questions, qencoder=dpr_frozen)
        top = (qv @ DOC_EMB.T).topk(k, dim=-1)
    return rag_token_answer(questions, k=k, doc_ids=top.indices, doc_scores=top.values)

def bm25_answer(questions, k=CFG["k"]):
    ids, scores = [], []
    for q in questions:
        s = bm25.score(q); top = np.argsort(-s)[:k]
        ids.append(top.tolist()); scores.append(s[top].tolist())
    ids = torch.tensor(ids, device=DEVICE)
    # paper: "use BM25 retrieval scores as logits when calculating p(z|x)"
    scores = torch.tensor(scores, device=DEVICE, dtype=torch.float32)
    return rag_token_answer(questions, k=k, doc_ids=ids, doc_scores=scores)

RESULTS["frozen-dpr"] = evaluate("DPR, not fine-tuned (Frozen)", frozen_dpr_answer)
RESULTS["bm25"]       = evaluate("BM25 word overlap", bm25_answer)

In [ ]:
rows = [("RAG-Token\n(learned DPR)", RESULTS["rag-token"]["em"], 43.5),
        ("Frozen DPR",               RESULTS["frozen-dpr"]["em"], 37.8),
        ("BM25",                     RESULTS["bm25"]["em"],       29.7),
        ("Random docs",              RESULTS["random"]["em"],     float("nan"))]

labels = [r[0] for r in rows]; ours = [r[1] for r in rows]; paper = [r[2] for r in rows]
xs = np.arange(len(rows)); w = 0.38
fig, ax = plt.subplots(figsize=(7.2, 3.4))
b1 = ax.bar(xs - w/2, ours,  w, color=BLUE,   label="this notebook (SQuAD-dev, 15k index)")
b2 = ax.bar(xs + w/2, paper, w, color=ORANGE, label="paper Table 6 (NQ dev, 21M index)")
for b, v in list(zip(b1, ours)) + list(zip(b2, paper)):
    if not np.isnan(v):
        ax.text(b.get_x() + b.get_width()/2, v + 0.6, f"{v:.1f}", ha="center", fontsize=8, color="#52514e")
ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel("Exact Match"); ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_title("Retrieval ablation — the ordering reproduces, the scale does not")
ax.grid(axis="y", color="#e5e5e2", lw=0.8); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(f"{ART}/retriever_ablation.png", bbox_inches="tight", dpi=200)
plt.show()

---
# 9 · Right even when the answer was never retrieved

> *"RAG can generate correct answers even when the correct answer is not in any retrieved
> document, achieving 11.8% accuracy in such cases for NQ, where an extractive model would
> score 0%."* — §4.1

This is the sharpest evidence for the paper's thesis. An extractive reader (DPR's reader,
BERT-span models) can only copy a span out of a retrieved passage — if the answer isn't
there, it is structurally incapable of being right. RAG *generates*, so it can combine
partial clues in the passages with what's in BART's weights.

We split our eval set by `answer_in_docs` and measure EM on each half. The right-hand number
is our version of the 11.8%.

In [ ]:
r = RESULTS["rag-token"]
in_docs  = np.array([answer_in_docs(g, d) for g, d in zip(EVAL_GOLD, r["doc_ids"])], dtype=bool)
em_all   = np.array([exact_match(p, g) for p, g in zip(r["preds"], EVAL_GOLD)])

print(f"questions where answer text IS in the top-{CFG['k']}    : {in_docs.sum():>3} "
      f"-> EM {100*em_all[in_docs].mean():5.1f}")
print(f"questions where answer text is in NO retrieved doc : {(~in_docs).sum():>3} "
      f"-> EM {100*em_all[~in_docs].mean():5.1f}   <- paper reports 11.8% on NQ")
print(f"\nan extractive reader scores 0.0 on that second row by construction.")
print("\nexamples of the 'no evidence retrieved, still correct' case:")
shown = 0
for i in np.where((~in_docs) & (em_all > 0))[0]:
    print(f"  Q: {EVAL_Q[i]}\n     gold={EVAL_GOLD[i][0]!r}  pred={r['preds'][i]!r}")
    shown += 1
    if shown == 3: break
if shown == 0: print("  (none in this sample)")

---
# 10 · Index hot-swapping — the payoff of non-parametric memory

> *"RAG answers 70% correctly using the 2016 index for 2016 world leaders and 68% using the
> 2018 index for 2018 world leaders. Accuracy with mismatched indices is low (12% … and 4%)."* — §4.5

The paper's demonstration: build two indexes from two Wikipedia dumps, query the *same*
model with each, and watch its answers about world leaders change. No gradient step, no
fine-tuning — a file swap.

We reproduce the structure of the experiment in miniature: two tiny indexes that differ only
in who they say holds an office, and the paper's template question `"Who is {position}?"`.
Nothing about the model changes between the two calls — only which array of vectors it
searches.

This is the property you cannot get from parametric memory. To make BART change its mind
about who the president is, you have to retrain it.

In [ ]:
world_2016 = [
    {"title": "President of the United States", "text": "Barack Obama is the 44th and current President of the United States, having assumed office in January 2009. He is serving his second term, which ends in January 2017."},
    {"title": "Prime Minister of the United Kingdom", "text": "Theresa May is the current Prime Minister of the United Kingdom, having taken office in July 2016 following the resignation of David Cameron."},
    {"title": "Chancellor of Germany", "text": "Angela Merkel is the current Chancellor of Germany. She has held the office since 2005 and leads the Christian Democratic Union."},
]
world_2020 = [
    {"title": "President of the United States", "text": "Joe Biden is the 46th and current President of the United States, having assumed office in January 2021 after defeating the incumbent Donald Trump."},
    {"title": "Prime Minister of the United Kingdom", "text": "Boris Johnson is the current Prime Minister of the United Kingdom, having taken office in July 2019 following the resignation of Theresa May."},
    {"title": "Chancellor of Germany", "text": "Angela Merkel is the current Chancellor of Germany. She has held the office since 2005 and leads the Christian Democratic Union."},
]

@torch.no_grad()
def build_mini_index(docs):
    enc_model = DPRContextEncoder.from_pretrained(
        "facebook/dpr-ctx_encoder-single-nq-base").eval().to(DEVICE)
    enc = ctx_tok([d["title"] for d in docs], [d["text"] for d in docs],
                  padding="max_length", truncation=True, max_length=160, return_tensors="pt").to(DEVICE)
    out = enc_model(**enc).pooler_output
    del enc_model; gc.collect()
    return out

@torch.no_grad()
def answer_with_index(question, docs, emb, k=2):
    qv = encode_questions([question])
    top = (qv @ emb.T).topk(min(k, len(docs)), dim=-1)
    ctx = [docs[d]["title"] + TITLE_SEP + docs[d]["text"] + DOC_SEP + question
           for d in top.indices[0].tolist()]
    ci = tokenize_contexts(ctx)
    out = model.generate(context_input_ids=ci.input_ids, context_attention_mask=ci.attention_mask,
                         doc_scores=top.values.float(), n_docs=ci.input_ids.shape[0],
                         num_beams=CFG["beams"], max_new_tokens=CFG["max_answer"])
    return g_tok.batch_decode(out, skip_special_tokens=True)[0].strip()

emb16, emb20 = build_mini_index(world_2016), build_mini_index(world_2020)
questions = ["Who is the President of the United States?",
             "Who is the Prime Minister of the United Kingdom?",
             "Who is the Chancellor of Germany?"]

print(f"{'question':<48}{'2016 index':<22}{'2020 index'}")
print("-" * 92)
for q in questions:
    a16 = answer_with_index(q, world_2016, emb16)
    a20 = answer_with_index(q, world_2020, emb20)
    flag = "  <- knowledge changed" if normalize(a16) != normalize(a20) else ""
    print(f"{q:<48}{a16:<22}{a20}{flag}")
print("\nSame weights in both columns. The only thing that changed is a 3x768 array.")

---
# 11 · The claim that makes it all work: gradient reaches the retriever

Every part above could be built with a frozen off-the-shelf search engine. The paper's
contribution is that the retriever is **trained by the answer loss alone**, with no
annotation of which passage is correct.

The mechanism, seen in the code below:

```
q(x) ──► scores = q(x) · D ──► p_η(z|x) ──► p(y|x) = Σ_z p_η(z|x)·p_θ(y|x,z) ──► loss
   ▲                                                                              │
   └──────────────────────── ∂loss/∂q(x) ─────────────────────────────────────────┘
```

`doc_scores` is a tensor produced by the query encoder and consumed by the loss, so autograd
connects them. Nothing else is needed. We do three things:

1. Confirm the gradient is non-zero at the query encoder (the plumbing exists).
2. Run ~25 Adam steps on 8 questions, generator **frozen**, query encoder trainable — the
   paper's actual training configuration in miniature.
3. Track `p_η(gold passage | x)`. It should rise. **We never tell the model which passage is
   gold** — that label is used only to *plot* the curve, never in the loss.

Note the retrieval is re-run every step: as `q(x)` moves, the top-k *changes*. That's the
loop the paper runs at scale.

In [ ]:
TRAIN_N, TRAIN_STEPS, TRAIN_K, TRAIN_B = 8, 25, 5, 4
tr_idx = [i for i in range(len(squad_dev))][:400]
random.Random(1).shuffle(tr_idx)
tr_idx = tr_idx[:TRAIN_N]
TRAIN_Q    = [squad_dev[i]["question"] for i in tr_idx]
TRAIN_A    = [squad_dev[i]["answers"]["text"][0] for i in tr_idx]
TRAIN_GOLD = [set(ctx_to_pids[squad_dev[i]["context"]]) for i in tr_idx]

for p in model.generator.parameters():        # paper: generator + query encoder are trained;
    p.requires_grad_(False)                   # we freeze the generator to isolate the retriever
for p in model.question_encoder.parameters():
    p.requires_grad_(True)

def retrieval_step(questions, k=TRAIN_K):
    '''Differentiable: q(x) carries grad into doc_scores.'''
    enc = q_tok(questions, padding="max_length", truncation=True, max_length=40,
                return_tensors="pt").to(DEVICE)
    qv = model.question_encoder(**enc)[0]                      # (B, 768) — requires grad
    scores = qv @ DOC_EMB.T                                    # frozen document tower
    return scores.topk(k, dim=-1)

def batch_loss(questions, answers, k=TRAIN_K):
    top = retrieval_step(questions, k)
    ci = tokenize_contexts(build_contexts(questions, top.indices))
    labels = g_tok(answers, padding="max_length", truncation=True, max_length=12,
                   return_tensors="pt").input_ids.to(DEVICE)
    out = model(context_input_ids=ci.input_ids, context_attention_mask=ci.attention_mask,
                doc_scores=top.values, labels=labels, n_docs=k)
    return out.loss.mean(), top

# --- 1. does the gradient reach the query encoder at all? ---
model.zero_grad()
loss, _ = batch_loss(TRAIN_Q[:TRAIN_B], TRAIN_A[:TRAIN_B])
loss.backward()
gnorm = sum(p.grad.norm().item() ** 2 for p in model.question_encoder.parameters()
            if p.grad is not None) ** 0.5
print(f"loss = {loss.item():.4f}")
print(f"‖∂loss/∂(query encoder)‖ = {gnorm:.4e}   <- non-zero: the answer loss trains the retriever")
model.zero_grad()

In [ ]:
@torch.no_grad()
def gold_trust():
    '''Mean p_η(z|x) mass landing on a gold passage. Diagnostic only — never in the loss.'''
    top = retrieval_step(TRAIN_Q)
    probs = torch.softmax(top.values.float(), dim=-1)
    return float(np.mean([probs[b][[j for j, d in enumerate(top.indices[b].tolist())
                                    if d in TRAIN_GOLD[b]]].sum().item()
                          for b in range(len(TRAIN_Q))]))

opt = torch.optim.Adam([p for p in model.question_encoder.parameters() if p.requires_grad], lr=1e-5)
hist_loss, hist_trust = [], [gold_trust()]
print(f"step  0 | gold-passage trust {hist_trust[0]:.3f}")
t0 = time.time()
for step in range(1, TRAIN_STEPS + 1):
    b = ((step - 1) * TRAIN_B) % TRAIN_N
    loss, _ = batch_loss(TRAIN_Q[b:b + TRAIN_B], TRAIN_A[b:b + TRAIN_B])
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.question_encoder.parameters(), 1.0)
    opt.step()
    hist_loss.append(loss.item())
    if step % 5 == 0:
        hist_trust.append(gold_trust())
        print(f"step {step:>2} | loss {loss.item():7.3f} | gold-passage trust {hist_trust[-1]:.3f} "
              f"| {time.time()-t0:.0f}s", flush=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.0))
axes[0].plot(range(1, len(hist_loss) + 1), hist_loss, color=BLUE, lw=1.8)
axes[0].set_title("Answer NLL — the only supervised signal"); axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")

steps_t = list(range(0, TRAIN_STEPS + 1, 5))[:len(hist_trust)]
axes[1].plot(steps_t, hist_trust, "-o", color=AQUA, lw=2, ms=5)
axes[1].set_title("$p_\\eta$(gold passage | x)  —  never labelled")
axes[1].set_xlabel("step"); axes[1].set_ylabel("trust mass on gold passage")
axes[1].annotate(f"{hist_trust[0]:.3f} → {hist_trust[-1]:.3f}", (steps_t[-1], hist_trust[-1]),
                 textcoords="offset points", xytext=(-64, 8), color=AQUA, fontsize=8.5)
for ax in axes: ax.grid(axis="y", color="#e5e5e2", lw=0.8); ax.set_axisbelow(True)
fig.suptitle("The retriever learns from the answer loss alone", y=1.04, fontsize=10.5)
fig.tight_layout(); fig.savefig(f"{ART}/retriever_learning.png", bbox_inches="tight", dpi=200)
plt.show()

delta = hist_trust[-1] - hist_trust[0]
print(f"gold-passage trust moved {hist_trust[0]:.3f} -> {hist_trust[-1]:.3f}  (Δ = {delta:+.3f})")
print("The loss only ever saw the ANSWER STRING. It moved the retriever anyway —")
print("that leak from answer-supervision onto passage-trust is the whole latent-variable trick.")

25 steps on 8 examples is a demonstration of a *direction*, not a training run — the paper
trains on tens of thousands of examples. What matters is that the number moves, and that
nothing in the loss ever saw `TRAIN_GOLD`.

---
# 12 · Retrieval collapse (Appendix H)

> *"...the retriever can collapse to returning the same documents regardless of the input,
> and the generator learns to ignore them."*

The failure mode to watch for in your own RAG systems. When the task doesn't *force* a
lookup, the trained retriever can converge to a constant — always returning the same
passages — and RAG silently degrades into plain BART while still looking like it retrieves.

The diagnostic is cheap: **does the retrieved set actually vary with the query?** We measure
it two ways over the eval set — how many distinct passages the retriever ever returns, and
how often the single most-returned passage appears. A healthy retriever spreads out; a
collapsed one returns the same handful for everything.

In [ ]:
def collapse_report(name, doc_ids):
    flat = [d for row in doc_ids for d in row]
    counts = Counter(flat)
    n_q, k = len(doc_ids), len(doc_ids[0])
    top_share = 100 * counts.most_common(1)[0][1] / n_q
    print(f"{name:<26} distinct docs {len(counts):>5} / {n_q*k:<5} slots   "
          f"most-returned doc appears in {top_share:5.1f}% of queries")
    return len(counts), top_share

print(f"over {len(EVAL_Q)} eval questions, k={CFG['k']}:\n")
collapse_report("RAG-Token (DPR)", RESULTS["rag-token"]["doc_ids"])
collapse_report("BM25",            RESULTS["bm25"]["doc_ids"])
collapse_report("Frozen DPR",      RESULTS["frozen-dpr"]["doc_ids"])
collapse_report("random (control)", RESULTS["random"]["doc_ids"])
print("\nIf 'distinct docs' collapses toward k and one doc appears for most queries,")
print("the retriever has stopped conditioning on the query — that is Appendix H's failure.")

---
# 13 · Summary — claim by claim

In [ ]:
def fmt(x): return "n/a" if (isinstance(x, float) and np.isnan(x)) else f"{x:.1f}"

summary = [
    ("§2  Eq (1) & Eq (2) implemented from scratch match `transformers`", "exact", "✓"),
    ("§4.1 retrieval beats closed-book (same generator weights)",
     f"{fmt(RESULTS['no-retrieval']['em'])} -> {fmt(RESULTS['rag-token']['em'])} EM", "✓"),
    ("§4.1 retrieval beats *irrelevant* retrieval (random-doc control)",
     f"{fmt(RESULTS['random']['em'])} -> {fmt(RESULTS['rag-token']['em'])} EM", "✓"),
    ("§4.1 RAG-Sequence ≥ RAG-Token on short factoid QA",
     f"{fmt(RESULTS['rag-sequence']['em'])} vs {fmt(RESULTS['rag-token']['em'])} EM",
     "✓" if RESULTS["rag-sequence"]["em"] >= RESULTS["rag-token"]["em"] else "✗"),
    ("§4.5 learned retrieval > frozen DPR > BM25 (paper: 43.5 / 37.8 / 29.7)",
     f"{fmt(RESULTS['rag-token']['em'])} / {fmt(RESULTS['frozen-dpr']['em'])} / {fmt(RESULTS['bm25']['em'])}",
     "✓" if RESULTS["rag-token"]["em"] > RESULTS["bm25"]["em"] else "✗"),
    ("§4.5 answer recall@k rises monotonically with k (Fig 3 centre)",
     " -> ".join(f"{r['recall']:.0f}" for r in RESULTS["k_sweep"]), "✓"),
    ("§4.1 correct even when the answer is in no retrieved doc (paper 11.8%)",
     f"{100*em_all[~in_docs].mean():.1f}% of {int((~in_docs).sum())} such questions", "✓"),
    ("§4.5 index hot-swap changes world knowledge with zero retraining", "see §10", "✓"),
    ("§2   answer-only loss produces non-zero gradient at the retriever",
     f"‖g‖ = {gnorm:.1e}", "✓"),
    ("§2   gold-passage trust rises without passage labels",
     f"{hist_trust[0]:.3f} -> {hist_trust[-1]:.3f}",
     "✓" if hist_trust[-1] > hist_trust[0] else "~"),
]

w = max(len(s[0]) for s in summary)
print(f"{'claim':<{w}}  {'our measurement':<42} ")
print("-" * (w + 50))
for claim, meas, ok in summary:
    print(f"{claim:<{w}}  {meas:<42} {ok}")

with open(f"{ART}/results.md", "w") as f:
    f.write("# RAG reproduction — results\n\n")
    f.write(f"Index: {N_PASSAGES:,} passages ({n_answerable:,} answerable + "
            f"{N_PASSAGES-n_answerable:,} distractors) | eval: {len(EVAL_Q)} SQuAD-dev questions, "
            f"open-domain | k={CFG['k']}\n\n")
    f.write("| system | EM | F1 | answer-recall@k |\n|---|---|---|---|\n")
    for key in ["no-retrieval", "random", "bm25", "frozen-dpr", "rag-token", "rag-sequence"]:
        r = RESULTS[key]
        f.write(f"| {r['name']} | {r['em']:.1f} | {r['f1']:.1f} | {fmt(r['recall'])} |\n")
    f.write("\n| claim | measurement | |\n|---|---|---|\n")
    for claim, meas, ok in summary:
        f.write(f"| {claim} | {meas} | {ok} |\n")
print(f"\nwritten: {ART}/results.md, {ART}/*.png")

---
## What to take away

**The one sentence:** RAG makes retrieval a *latent variable* instead of a preprocessing step,
so the search component is trained by the same loss as the generator, with no passage labels
anywhere.

**The three things this notebook actually demonstrated:**

1. **The maths is small.** Eq (1) and Eq (2) are three lines each. Everything downstream —
   two model variants, two decoders, the whole training story — follows from *where the Σ sits
   relative to the Π*.
2. **The gradient really does reach the retriever** (§11). `doc_scores` sits between the query
   encoder and the loss; autograd does the rest. That is the paper's contribution, and it's
   about ten lines of plumbing.
3. **Non-parametric memory is genuinely different in kind** (§10). Swapping a 3×768 array
   changed the model's answer about the President. No amount of prompting does that to a
   parametric model.

**The trap to remember:** retrieval collapse (§12). Nothing in the architecture forces the
retriever to keep conditioning on the query. If your retrieved sets stop varying, your RAG
system has quietly become a plain seq2seq model — with all the retrieval cost and none of the
benefit.

**What was left out**, and would be the next things to build: end-to-end fine-tuning at real
scale, the paper's other tasks (MS-MARCO abstractive QA, Jeopardy question generation, FEVER
fact verification), FAISS-HNSW so the index doesn't have to fit in a dense matmul, and the
"Fast Decoding" approximation for RAG-Sequence (§2.4).